# CAMUS Echocardiography Segmentation in Google Colab

This notebook clones the GitHub repo, mounts Google Drive, installs Colab-safe dependencies, runs tests, trains a model, evaluates it, and saves outputs back to Drive.

## 1. Use a GPU Runtime

In Colab, choose **Runtime -> Change runtime type -> GPU** before running the training cells.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
REPO_URL = 'https://github.com/yasser2652/Deep-Echocardiographic-segmentation-.git'
PROJECT_DIR = '/content/DeepEchoSeg'

%cd /content
!rm -rf "$PROJECT_DIR"
!git clone "$REPO_URL" "$PROJECT_DIR"
%cd "$PROJECT_DIR"

!pwd
!python -c "import src; print('src import OK')"

In [ ]:
# Colab usually already has CUDA-enabled torch installed. Avoid reinstalling torch unless you know you need to.
!pip install -q SimpleITK nibabel albumentations pyyaml tqdm pandas scipy scikit-image matplotlib pytest

In [ ]:
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

## 2. Configure CAMUS Paths

Put CAMUS in Google Drive, usually at `/content/drive/MyDrive/CAMUS`. If you uploaded `CAMUS.zip`, uncomment the unzip cell below.

In [ ]:
DATA_ROOT = '/content/drive/MyDrive/CAMUS'
OUTPUT_DIR = '/content/drive/MyDrive/camus_outputs'
RUN_NAME = 'baseline_unet_colab'

print('DATA_ROOT:', DATA_ROOT)
print('OUTPUT_DIR:', OUTPUT_DIR)

In [ ]:
# Optional: unzip CAMUS.zip from Drive if needed.
# CAMUS_ZIP = '/content/drive/MyDrive/CAMUS.zip'
# !unzip -q "$CAMUS_ZIP" -d /content/drive/MyDrive

## 3. Run Tests

In [ ]:
!PYTHONDONTWRITEBYTECODE=1 pytest -q tests

## 4. Optional Smoke Training Run

Run this first to verify the training loop in Colab without CAMUS.

In [ ]:
!python -m src.train --config config.yaml --create-dummy-data --epochs 1 --batch-size 2 --image-size 64 --model baseline_unet --output-dir "$OUTPUT_DIR" --run-name smoke_colab --device "$DEVICE"

## 5. Train on CAMUS

Start with a small epoch count, then increase after confirming paths and GPU memory.

In [ ]:
EPOCHS = 100
BATCH_SIZE = 8
IMAGE_SIZE = 256
MODEL = 'baseline_unet'

!python -m src.train \
  --config config.yaml \
  --data-root "$DATA_ROOT" \
  --output-dir "$OUTPUT_DIR" \
  --run-name "$RUN_NAME" \
  --model "$MODEL" \
  --epochs "$EPOCHS" \
  --batch-size "$BATCH_SIZE" \
  --image-size "$IMAGE_SIZE" \
  --device "$DEVICE" \
  --mixed-precision

## 6. Evaluate

In [ ]:
CHECKPOINT = f'{OUTPUT_DIR}/{RUN_NAME}/best.pth'
EVAL_DIR = f'{OUTPUT_DIR}/{RUN_NAME}/evaluation'

!python -m src.evaluate \
  --checkpoint "$CHECKPOINT" \
  --data-root "$DATA_ROOT" \
  --output-dir "$EVAL_DIR" \
  --split test \
  --device "$DEVICE"

## 7. Predict a Patient Folder

In [ ]:
# Change this to an actual patient folder after CAMUS is mounted.
PATIENT_DIR = f'{DATA_ROOT}/patient0001'
PRED_DIR = f'{OUTPUT_DIR}/{RUN_NAME}/predictions_patient0001'

!python -m src.predict \
  --checkpoint "$CHECKPOINT" \
  --input "$PATIENT_DIR" \
  --output-dir "$PRED_DIR" \
  --device "$DEVICE" \
  --save-confidence \
  --postprocess

## 8. Visualize One Prediction

This displays the original image, ground-truth mask, predicted mask, overlays, and an error map directly in Colab output. Change `VIS_IMAGE_PATH` to inspect another case.

In [ ]:
%cd /content/DeepEchoSeg

import torch
import numpy as np
from pathlib import Path
from PIL import Image, ImageDraw
from IPython.display import display

from src.dataset import load_medical_image, select_2d
from src.transforms import SegmentationTransform
from src.model_registry import build_model_from_config
from src.utils import load_checkpoint
from src.postprocess import postprocess_mask
from src.metrics import dice_per_class

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Change these if you trained a different run or want to inspect another patient/view/phase.
VIS_RUN_NAME = RUN_NAME
VIS_CHECKPOINT = f'{OUTPUT_DIR}/{VIS_RUN_NAME}/best.pth'
VIS_IMAGE_PATH = f'{DATA_ROOT}/patient0500/patient0500_2CH_ED.nii.gz'

p = Path(VIS_IMAGE_PATH)
if p.name.endswith('.nii.gz'):
    VIS_GT_PATH = str(p.with_name(p.name.replace('.nii.gz', '_gt.nii.gz')))
else:
    VIS_GT_PATH = str(p.with_name(p.stem + '_gt' + p.suffix))

print('Checkpoint exists:', Path(VIS_CHECKPOINT).exists(), VIS_CHECKPOINT)
print('Image exists:', Path(VIS_IMAGE_PATH).exists(), VIS_IMAGE_PATH)
print('GT exists:', Path(VIS_GT_PATH).exists(), VIS_GT_PATH)

ckpt = load_checkpoint(VIS_CHECKPOINT, map_location=DEVICE)
config = ckpt['config']

model = build_model_from_config(config).to(DEVICE)
model.load_state_dict(ckpt['model_state'])
model.eval()

image_np = select_2d(load_medical_image(VIS_IMAGE_PATH))
gt_np = select_2d(load_medical_image(VIS_GT_PATH))

transform = SegmentationTransform(
    image_size=int(config.get('image_size', 256)),
    training=False,
    normalize=config.get('preprocessing', {}).get('normalize', 'minmax'),
    z_score=bool(config.get('preprocessing', {}).get('z_score', False)),
)

image_tensor, _ = transform(image_np, np.zeros_like(image_np, dtype=np.int64))
_, gt_tensor = transform(image_np, gt_np)
gt = gt_tensor.numpy().astype(np.uint8)

temporal_window = int(config.get('temporal_window', 1))
if temporal_window > image_tensor.shape[0]:
    image_tensor = image_tensor.repeat(temporal_window, 1, 1)

with torch.no_grad():
    x = image_tensor.unsqueeze(0).to(DEVICE)
    logits = model(x)
    pred = torch.softmax(logits, dim=1).argmax(dim=1)[0].cpu().numpy().astype(np.uint8)

pred = postprocess_mask(
    pred,
    num_classes=int(config.get('num_classes', 4)),
    config=config.get('postprocessing', {}),
).astype(np.uint8)

def to_uint8_gray(x):
    x = np.asarray(x, dtype=np.float32)
    x = x - x.min()
    x = x / (x.max() + 1e-8)
    return (x * 255).astype(np.uint8)

palette = np.array([
    [0, 0, 0],        # background
    [230, 57, 70],   # LV cavity
    [42, 157, 143],  # myocardium
    [69, 123, 157],  # left atrium
], dtype=np.uint8)

def colorize(mask):
    out = np.zeros((*mask.shape, 3), dtype=np.uint8)
    for cls in range(min(len(palette), int(mask.max()) + 1)):
        out[mask == cls] = palette[cls]
    return out

def overlay(gray_u8, mask, alpha=0.45):
    rgb = np.repeat(gray_u8[..., None], 3, axis=-1)
    color = colorize(mask)
    fg = mask > 0
    out = rgb.copy()
    out[fg] = ((1 - alpha) * rgb[fg] + alpha * color[fg]).astype(np.uint8)
    return out

image_display = image_tensor[image_tensor.shape[0] // 2].cpu().numpy()
gray = to_uint8_gray(image_display)

gt_color = colorize(gt)
pred_color = colorize(pred)
gt_overlay = overlay(gray, gt)
pred_overlay = overlay(gray, pred)

error_map = np.repeat(gray[..., None], 3, axis=-1)
correct = (gt == pred) & (gt > 0)
false_pred = (gt != pred) & (pred > 0)
missed_gt = (gt != pred) & (gt > 0)
error_map[correct] = [0, 200, 0]
error_map[false_pred] = [255, 0, 0]
error_map[missed_gt] = [255, 220, 0]

dice = dice_per_class(pred[None], gt[None], num_classes=int(config.get('num_classes', 4)))
print('Dice:')
print('  LV cavity:', round(float(dice[1]), 4))
print('  myocardium:', round(float(dice[2]), 4))
print('  left atrium:', round(float(dice[3]), 4))
print('  mean foreground:', round(float(np.mean(dice[1:])), 4))

# Build one display image so Colab shows it reliably, even when matplotlib backends are non-interactive.
tiles = [
    ('Original image', np.repeat(gray[..., None], 3, axis=-1)),
    ('Ground-truth mask', gt_color),
    ('Predicted mask', pred_color),
    ('Ground-truth overlay', gt_overlay),
    ('Prediction overlay', pred_overlay),
    ('Error map', error_map),
]

tile_w, tile_h = 256, 256
label_h = 34
cols, rows = 3, 2
canvas = Image.new('RGB', (cols * tile_w, rows * (tile_h + label_h)), 'white')
draw = ImageDraw.Draw(canvas)

for i, (title, arr) in enumerate(tiles):
    r = i // cols
    c = i % cols
    img = Image.fromarray(arr).resize((tile_w, tile_h), Image.Resampling.NEAREST)
    x = c * tile_w
    y = r * (tile_h + label_h)
    draw.text((x + 8, y + 8), title, fill='black')
    canvas.paste(img, (x, y + label_h))

display(canvas)
